In [ ]:
!pip install -q transformers==4.44.2 accelerate>=1.0 sentence-transformers

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

NOME_MODELLO = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(NOME_MODELLO)
model = AutoModelForCausalLM.from_pretrained(
    NOME_MODELLO,
    torch_dtype=torch.float16,
    device_map="auto",
)

In [ ]:
import torch.nn.functional as F
class ModelloPerplexity:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.device = model.device

    @torch.no_grad()
    def logprob_testo(self, testo_nodo, contesto=""):
        if contesto:
            prompt = contesto + "\n\n" + testo_nodo
            n_ctx_tok = self.tokenizer(contesto + "\n\n", return_tensors="pt").input_ids.shape[1]
        else:
            prompt = testo_nodo
            n_ctx_tok = 1

        ids = self.tokenizer(prompt, return_tensors="pt").input_ids.to(self.device)
        logits = self.model(ids).logits

        logprobs = F.log_softmax(logits[0, :-1], dim=-1)
        target = ids[0, 1:]
        tok_logprob = logprobs[torch.arange(target.shape[0]), target]

        nodo_logprob = tok_logprob[n_ctx_tok - 1:]
        somma = nodo_logprob.sum().item()
        n_token = nodo_logprob.shape[0]
        return somma, n_token

    def perplexity_testo(self, testo_nodo, contesto=""):
        somma, n_token = self.logprob_testo(testo_nodo, contesto)
        if n_token == 0:
            return float("inf")
        return float(torch.exp(torch.tensor(-somma / n_token)))

In [ ]:
m = ModelloPerplexity(model, tokenizer)

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx

BASE = "/kaggle/input/datasets/angelo01paldino/datasetcora"

# ordine nodi da cora.content
content = pd.read_csv(f"{BASE}/cora.content", sep="\t", header=None)
paper_ids = content.iloc[:, 0].astype(str).values
n_nodi = len(paper_ids)
id_to_index = {pid: i for i, pid in enumerate(paper_ids)}

# archi da cora.cites
cites = pd.read_csv(f"{BASE}/cora.cites", sep="\t", header=None, names=["cited", "citing"], dtype=str)
G = nx.Graph()
G.add_nodes_from(range(n_nodi))
for cited, citing in zip(cites["cited"], cites["citing"]):
    if cited in id_to_index and citing in id_to_index:
        G.add_edge(id_to_index[citing], id_to_index[cited])

# testo + etichette dal CSV combinato, ordinato per id
tape = pd.read_csv(f"{BASE}/combined.csv").sort_values("id").reset_index(drop=True)

testi = []
for i in range(n_nodi):
    t = tape.loc[i, "T"]; a = tape.loc[i, "A"]
    t = "" if pd.isna(t) else str(t).strip()
    a = "" if pd.isna(a) else str(a).strip()
    if t and a: testi.append(f"{t}. {a}")
    elif t: testi.append(t)
    elif a: testi.append(a)
    else: testi.append("")

etichette = tape["label"].values.astype(int)
print(f"Nodi: {n_nodi}, Archi: {G.number_of_edges()}, Classi: {len(set(etichette))}")

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

testi_per_embedding = [t[:1000] if t else "" for t in testi]
embeddings = embedder.encode(testi_per_embedding, convert_to_numpy=True, show_progress_bar=True, batch_size=64)
print("Shape embeddings:", embeddings.shape)  # (2708, 384)

In [ ]:
import numpy as np
from tqdm import tqdm


class ScorerPerplexity:
    def __init__(self, modello, testi, embeddings, max_nodi_contesto=20, max_char_testo=300):
        self.modello = modello
        self.testi = testi
        self.embeddings = embeddings
        self.max_nodi_contesto = max_nodi_contesto
        self.max_char_testo = max_char_testo

    def _costruisci_contesto(self, nodo, membri_community):
        altri = [m for m in membri_community if m != nodo]
        if not altri:
            return ""
        # centroide semantico della community (esclusi il nodo stesso)
        centroide = self.embeddings[altri].mean(axis=0)
        # scelgo i nodi piu' vicini al centroide (cosine similarity)
        emb_altri = self.embeddings[altri]
        sim = emb_altri @ centroide / (
            np.linalg.norm(emb_altri, axis=1) * np.linalg.norm(centroide) + 1e-8
        )
        k = min(self.max_nodi_contesto, len(altri))
        idx_top = np.argsort(sim)[::-1][:k]
        selezionati = [altri[i] for i in idx_top]
        pezzi = [self.testi[m][:self.max_char_testo] for m in selezionati if self.testi[m]]
        return "\n\n".join(pezzi)

    def perplexity_nodo(self, nodo, membri_community):
        testo = self.testi[nodo][:self.max_char_testo]
        if not testo:
            return None
        contesto = self._costruisci_contesto(nodo, membri_community)
        return self.modello.perplexity_testo(testo, contesto)

    def ppl_partizione(self, partizione, mostra_avanzamento=True, descrizione="PPL(C)"):
        community = {}
        for nodo, c in enumerate(partizione):
            community.setdefault(c, []).append(nodo)

        ppl_per_nodo = {}
        iteratore = range(len(partizione))
        if mostra_avanzamento:
            iteratore = tqdm(iteratore, desc=descrizione, unit="nodo")

        for nodo in iteratore:
            c = partizione[nodo]
            ppl = self.perplexity_nodo(nodo, community[c])
            if ppl is not None:
                ppl_per_nodo[nodo] = ppl

        if not ppl_per_nodo:
            return float("inf"), {}
        media = float(np.mean(list(ppl_per_nodo.values())))
        return media, ppl_per_nodo

In [ ]:
import numpy as np

def _seleziona_contesto_da_sorgente(self, nodo_target, membri_sorgente):
    """
    Costruisce il contesto per predire nodo_target usando SOLO i nodi di
    membri_sorgente (che possono appartenere a un'altra community).
    Sceglie i piu' vicini semanticamente al nodo target.
    """
    altri = [m for m in membri_sorgente if m != nodo_target]
    if not altri:
        return ""
    emb_target = self.embeddings[nodo_target]
    emb_altri = self.embeddings[altri]
    sim = emb_altri @ emb_target / (
        np.linalg.norm(emb_altri, axis=1) * np.linalg.norm(emb_target) + 1e-8
    )
    k = min(self.max_nodi_contesto, len(altri))
    idx_top = np.argsort(sim)[::-1][:k]
    selezionati = [altri[i] for i in idx_top]
    pezzi = [self.testi[m][:self.max_char_testo] for m in selezionati if self.testi[m]]
    return "\n\n".join(pezzi)

def perplexity_nodo_con_sorgente(self, nodo_target, membri_sorgente):
    """
    Perplessita' del testo di nodo_target predetto col contesto di membri_sorgente.
    Se sorgente == community del nodo -> perplessita' interna.
    Se sorgente == altra community    -> perplessita' incrociata.
    """
    testo = self.testi[nodo_target][:self.max_char_testo]
    if not testo:
        return None
    contesto = self._seleziona_contesto_da_sorgente(nodo_target, membri_sorgente)
    return self.modello.perplexity_testo(testo, contesto)

ScorerPerplexity._seleziona_contesto_da_sorgente = _seleziona_contesto_da_sorgente
ScorerPerplexity.perplexity_nodo_con_sorgente = perplexity_nodo_con_sorgente

In [ ]:
scorer = ScorerPerplexity(m, testi, embeddings)


## `AggregazionePerplexity` — Fusione di community adiacenti tramite perplexity incrociata

Fonde due community adiacenti solo se sono lo stesso topic, misurato con perplessità incrociata: il contesto di A deve predire bene i nodi di B e viceversa. Il confronto è sempre "predici B con A" contro "predici B con B" a parità di nodi target, così da eliminare il bias dimensionale (più contesto → ppl più bassa).

### `__init__(scorer, grafo, partizione, soglia_relativa=0.05, max_passate=3, max_campione=15, seed=42, dir_salvataggio="/kaggle/working")`

- `scorer`: oggetto che espone `perplexity_nodo_con_sorgente`.
- `self.partizione`: copia in `np.array` della partizione in ingresso.
- `soglia_relativa`: degrado incrociato tollerato (`0.05` = 5%).
- `max_passate`: numero massimo di passate di aggregazione.
- `max_campione`: dimensione massima del campione di nodi target.
- `self.rng`: generatore `np.random.default_rng(seed)` per il campionamento.

### Metodi ausiliari

**`_mappa_community_membri()`**
1. Costruisce il dizionario `{id_community -> lista di nodi}` scorrendo `self.partizione` con `setdefault`.

**`_campiona(membri)`**
2. Se `len(membri) > max_campione`, estrae un campione senza reinserimento di dimensione `max_campione`; altrimenti restituisce tutti i membri.

**`_ppl_media_target_su_sorgente(membri_target, membri_sorgente)`**
3. Campiona i nodi target.
4. Per ciascuno calcola `scorer.perplexity_nodo_con_sorgente(nodo, membri_sorgente)`, scartando i `None`.
5. Ritorna la media dei valori raccolti, oppure `float("inf")` se la lista è vuota.

**`_coppie_community_adiacenti()`**
6. Scorre gli archi del grafo; per ogni arco che collega community diverse aggiunge la coppia ordinata `(min, max)` a un `set`, evitando duplicati.

### `_aggrega_una_passata()`

7. Ottiene le coppie adiacenti e la mappa community→membri; inizializza `fusioni = 0`.
8. Per ogni coppia `(ca, cb)` (con barra `tqdm`): se una delle due non è più nella mappa (già fusa), la salta.
9. **Perplessità interna**: `ppl_a_interna` (A predetto con A) e `ppl_b_interna` (B con B); si media nelle due direzioni → `ppl_interna`.
10. **Perplessità incrociata**: `ppl_a_su_b` (A predetto con B) e `ppl_b_su_a` (B con A); si media → `ppl_incrociata`.
11. **Criterio di fusione**: se `ppl_incrociata <= ppl_interna * (1 + soglia_relativa)`, ovvero il contesto altrui predice quasi bene quanto quello proprio, allora:
    - tutti i nodi di `cb` vengono riassegnati a `ca`;
    - `mappa[ca]` diventa l'unione dei membri, `mappa[cb]` viene rimossa;
    - `fusioni += 1`.
12. Ritorna il numero di fusioni effettuate.

### `ottimizza()`

13. **Ciclo esterno** su `max_passate`: per ciascuna registra il numero di community prima (`n_prima`), esegue `_aggrega_una_passata()` e conta le community dopo (`n_dopo`).
14. Stampa il riepilogo e appende a `storia` il dizionario `{passata, fusioni, n_community}`.
15. **Salvataggio su disco**: `partizione_aggregata_passata_{i}.npy` dopo ogni passata.
16. **Criterio di arresto**: se `fusioni == 0` la convergenza è raggiunta e il ciclo termina anticipatamente.
17. **Ritorno**: `(self.partizione, storia)`.

In [ ]:
import numpy as np
from tqdm import tqdm


class AggregazionePerplexity:
    """
    Fonde due community adiacenti SOLO se sono lo stesso topic, misurato con
    perplessita' incrociata: il contesto di A deve predire bene i nodi di B e
    viceversa. Elimina il bias dimensionale (piu' contesto -> ppl piu' bassa)
    perche' confronta sempre "predici B con A" contro "predici B con B" a parita'
    di nodi target.
    """

    def __init__(self, scorer, grafo, partizione, soglia_relativa=0.05,
                 max_passate=3, max_campione=15, seed=42,
                 dir_salvataggio="/kaggle/working"):
        self.scorer = scorer
        self.grafo = grafo
        self.partizione = np.array(partizione).copy()
        self.soglia_relativa = soglia_relativa   # quanto degrado incrociato tollero (0.05 = 5%)
        self.max_passate = max_passate
        self.max_campione = max_campione
        self.rng = np.random.default_rng(seed)
        self.dir_salvataggio = dir_salvataggio

    def _mappa_community_membri(self):
        m = {}
        for nodo, c in enumerate(self.partizione):
            m.setdefault(c, []).append(nodo)
        return m

    def _campiona(self, membri):
        if len(membri) > self.max_campione:
            return list(self.rng.choice(membri, size=self.max_campione, replace=False))
        return list(membri)

    def _ppl_media_target_su_sorgente(self, membri_target, membri_sorgente):
        """Media della perplessita' dei target predetti col contesto della sorgente."""
        campione = self._campiona(membri_target)
        valori = []
        for nodo in campione:
            ppl = self.scorer.perplexity_nodo_con_sorgente(nodo, membri_sorgente)
            if ppl is not None:
                valori.append(ppl)
        return float(np.mean(valori)) if valori else float("inf")

    def _coppie_community_adiacenti(self):
        coppie = set()
        for u, v in self.grafo.edges():
            cu, cv = self.partizione[u], self.partizione[v]
            if cu != cv:
                coppie.add((min(cu, cv), max(cu, cv)))
        return coppie

    def _aggrega_una_passata(self):
        fusioni = 0
        coppie = self._coppie_community_adiacenti()
        mappa = self._mappa_community_membri()

        for (ca, cb) in tqdm(list(coppie), desc="aggregazione", unit="coppia"):
            if ca not in mappa or cb not in mappa:
                continue
            membri_a, membri_b = mappa[ca], mappa[cb]

            # perplessita' interna: predico A con A, B con B
            ppl_a_interna = self._ppl_media_target_su_sorgente(membri_a, membri_a)
            ppl_b_interna = self._ppl_media_target_su_sorgente(membri_b, membri_b)
            ppl_interna = (ppl_a_interna + ppl_b_interna) / 2

            # perplessita' incrociata: predico A con B, B con A
            ppl_a_su_b = self._ppl_media_target_su_sorgente(membri_a, membri_b)
            ppl_b_su_a = self._ppl_media_target_su_sorgente(membri_b, membri_a)
            ppl_incrociata = (ppl_a_su_b + ppl_b_su_a) / 2

            # fondo solo se il contesto dell'altra community predice quasi bene
            # quanto il contesto proprio (entro la soglia relativa)
            if ppl_incrociata <= ppl_interna * (1 + self.soglia_relativa):
                for nodo in membri_b:
                    self.partizione[nodo] = ca
                mappa[ca] = membri_a + membri_b
                del mappa[cb]
                fusioni += 1

        return fusioni

    def ottimizza(self):
        storia = []
        for passata in range(self.max_passate):
            n_prima = len(set(self.partizione))
            fusioni = self._aggrega_una_passata()
            n_dopo = len(set(self.partizione))
            print(f"Passata {passata+1}: {n_prima} -> {n_dopo} community ({fusioni} fusioni)")
            storia.append({"passata": passata+1, "fusioni": int(fusioni), "n_community": int(n_dopo)})
            np.save(f"{self.dir_salvataggio}/partizione_aggregata_passata_{passata+1}.npy", self.partizione)
            if fusioni == 0:
                print("Nessuna fusione: convergenza raggiunta.")
                break
        return self.partizione, storia

In [ ]:
# carico la partizione gia' prodotta dal local-moving
partizione_lm = np.load("/kaggle/input/datasets/angelo01paldino/partizione3/partizione_aggregata_passata_2 (1).npy")

assert len(partizione_lm) == G.number_of_nodes()

agg = AggregazionePerplexity(
    scorer, G, partizione_lm,
    soglia_relativa=0.05,   # quanto degrado incrociato tollero per fondere (5%)
    max_passate=3,
    max_campione=25         # nodi campionati per stimare la perplessita'
)
partizione_aggregata, storia = agg.ottimizza()

import json
np.save("/kaggle/working/partizione_aggregata_finale.npy", partizione_aggregata)
with open("/kaggle/working/storia_aggregazione.json", "w") as f:
    json.dump(storia, f, indent=2)